# 面试题：如何从零实现 DiffPool，并用于层次化图分类？

## 面试回答主线

DiffPool 同时学习节点表示 Z=GNN_embed(A,X) 和软聚类矩阵 S=softmax(GNN_pool(A,X))，再计算 X'=S^T Z、A'=S^TAS，把节点级图压缩成更小的簇级图。与全局 mean pooling 相比，它能保留“哪些节点共同组成子结构”以及簇之间的连接。实现时要关注邻接归一化、批量矩阵形状、assignment 是否塌缩、链接预测或熵辅助损失和 padding mask。节点置换后的输出应保持不变，但“同一模板换编号”只能验证不变量，不能证明模型能泛化到新结构。评估应把模板重排和真正未见拓扑分开报告，分类阈值只能由训练约定或训练校准集决定，不能看测试标签后挑选。下面不用 PyG、DGL 或现成 DiffPool，而是用 PyTorch 矩阵乘法与 nn.Parameter 手写完整 forward、真实 backward、层次粗化、未见结构评估和 assignment 故障修复。

## 真实案例：支付交易子图识别高风险闭环

每张图固定为 6 个脱敏账户节点，节点特征只有风险分和归一化交易额。训练集由两种基础拓扑的 6 组配对图组成：每一对正常图与闭环图拥有完全相同的节点特征、节点排列和边数，只改变高风险账户之间是否形成闭环。模板重排集继续使用训练拓扑，但换金额微扰与节点编号，用于测量同模板稳定性。未见结构集包含 4 对从未参与训练的拓扑，边数分别为 5、7、6、8，其中正常类覆盖高风险路径、低风险桥和稠密高低风险二部连接，正类覆盖三角闭环、四节点闭环和不同尾链。每个未见配对仍共享完全相同的节点特征和边数，因此标签不能从均值、节点顺序或总边数泄漏；真正可用的信号只有关系组合。数据是机制教学用的离线结构，不代表真实反欺诈效果。

In [1]:
import math  # 导入平方根用于手写参数初始化。
import warnings  # 导入告警控制工具以保持保存输出聚焦教学结果。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 只过滤当前环境在导入 PyTorch 时产生的第三方弃用提示。
import torch  # 导入 PyTorch 以构造图张量并真实训练。
from torch import nn  # 导入基础模块与可学习参数。
import torch.nn.functional as F  # 导入底层交叉熵。
torch.set_num_threads(1)  # 固定小图实验为单线程以减少运行波动。
torch.manual_seed(61)  # 固定参数初始化和图特征扰动。
label_names = ["无高风险闭环", "高风险闭环"]  # 定义只描述关系结构的两类名称。
base_amount = torch.tensor([0.62, 0.55, 0.70, 0.40, 0.48, 0.58])  # 定义两类配对图共同使用的金额基础值。
training_risk = torch.tensor([0.92, 0.86, 0.80, 0.18, 0.12, 0.10])  # 定义训练模板的三高三低风险账户。
training_edges = {  # 定义训练阶段唯一可见的两种拓扑模板。
    0: [(0, 3), (1, 4), (2, 5), (3, 4), (4, 5), (5, 3)],  # 正常模板把三个高风险账户彼此隔开。
    1: [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 5)],  # 闭环模板让三个高风险账户形成三角形。
}  # 完成同为六条边的训练模板。
def make_features(risk_profile, seed):  # 根据共享风险画像生成与标签无关的节点特征。
    generator = torch.Generator().manual_seed(seed)  # 为当前配对图建立确定性随机源。
    amount_noise = torch.randn(6, generator=generator) * 0.015  # 生成只由配对编号决定的金额微扰。
    amounts = base_amount + amount_noise  # 得到同一配对两类完全共享的金额字段。
    return torch.stack([risk_profile, amounts], dim=-1)  # 返回六节点两字段特征矩阵。
def adjacency_from_edges(edges):  # 把无向边列表转换成六乘六邻接矩阵。
    adjacency = torch.zeros(6, 6)  # 创建不含自环的初始邻接矩阵。
    for source, target in edges:  # 遍历当前拓扑中的全部边。
        adjacency[source, target] = 1.0  # 写入正向邻接位置。
        adjacency[target, source] = 1.0  # 写入反向位置以表示无向交易关系。
    return adjacency  # 返回可用于消息传递的邻接矩阵。
def permute_graph(features, adjacency, seed):  # 对节点特征和邻接行列应用同一确定性置换。
    generator = torch.Generator().manual_seed(seed)  # 为当前配对建立独立置换随机源。
    permutation = torch.randperm(6, generator=generator)  # 生成六个节点的新编号顺序。
    permuted_features = features[permutation]  # 按新编号重排节点特征。
    permuted_adjacency = adjacency[permutation][:, permutation]  # 同步重排邻接矩阵行列。
    return permuted_features, permuted_adjacency, permutation  # 返回语义不变的重排图。
def make_paired_graphs(prefix, pair_count, feature_seed_start, permutation_seed_start):  # 生成训练模板的成对图集合。
    paired_graphs = []  # 收集当前集合中的正常与闭环配对。
    for pair_index in range(pair_count):  # 为每个特征版本建立一对标签图。
        shared_features = make_features(training_risk, feature_seed_start + pair_index)  # 生成两类共同使用的节点特征。
        for label in (0, 1):  # 使用相同特征分别实例化两种关系模板。
            adjacency = adjacency_from_edges(training_edges[label])  # 根据类别选择关系结构而不改变边数。
            features, adjacency, permutation = permute_graph(shared_features, adjacency, permutation_seed_start + pair_index)  # 对配对图使用完全相同的节点置换。
            graph_id = f"{prefix}-{pair_index + 1}{'N' if label == 0 else 'R'}"  # 构造可读的配对图编号。
            paired_graphs.append({"id": graph_id, "pair": pair_index, "features": features, "adjacency": adjacency, "label": label, "topology": "训练模板重排"})  # 保存图张量与结构来源。
    return paired_graphs  # 返回特征和边数均严格配对的图集合。
train_graphs = make_paired_graphs("TR", 6, 300, 800)  # 创建十二张训练模板图。
template_graphs = make_paired_graphs("TP", 3, 500, 1000)  # 创建六张新编号新微扰的模板重排图。
unseen_specs = [  # 定义四组训练阶段从未出现的结构配对。
    {"name": "五边低风险桥/三角尾链", "risk": [0.92, 0.86, 0.80, 0.18, 0.12, 0.10], "normal": [(0, 3), (1, 3), (2, 4), (3, 4), (4, 5)], "ring": [(0, 1), (1, 2), (2, 0), (2, 3), (4, 5)]},  # 用五边稀疏结构测试三角闭环加短尾。
    {"name": "七边高风险路径/三角多桥", "risk": [0.92, 0.86, 0.80, 0.18, 0.12, 0.10], "normal": [(0, 1), (1, 2), (0, 3), (2, 4), (3, 4), (4, 5), (5, 3)], "ring": [(0, 1), (1, 2), (2, 0), (0, 3), (2, 4), (4, 5), (5, 3)]},  # 区分高风险路径与闭合三角形。
    {"name": "六边四高路径/四节点闭环", "risk": [0.92, 0.86, 0.80, 0.74, 0.12, 0.10], "normal": [(0, 1), (1, 2), (2, 3), (0, 4), (3, 5), (4, 5)], "ring": [(0, 1), (1, 2), (2, 3), (3, 0), (0, 4), (2, 5)]},  # 用四个高风险节点测试训练未见的四边闭环。
    {"name": "八边高低二部/稠密三角", "risk": [0.92, 0.86, 0.80, 0.18, 0.12, 0.10], "normal": [(0, 3), (0, 4), (1, 3), (1, 5), (2, 4), (2, 5), (3, 4), (4, 5)], "ring": [(0, 1), (1, 2), (2, 0), (0, 3), (1, 4), (2, 5), (3, 4), (4, 5)]},  # 用相同八边预算比较二部连接与高风险闭环。
]  # 完成不同边数和不同闭环形态的未见结构定义。
unseen_graphs = []  # 收集八张真正未见拓扑的留出图。
for pair_index, specification in enumerate(unseen_specs):  # 遍历四个未见结构配对。
    risk_profile = torch.tensor(specification["risk"])  # 读取当前配对共享的风险分集合。
    shared_features = make_features(risk_profile, 700 + pair_index)  # 生成与标签无关的共享金额微扰。
    for label, edge_key in ((0, "normal"), (1, "ring")):  # 分别创建无闭环与有闭环图。
        adjacency = adjacency_from_edges(specification[edge_key])  # 从当前未见边表构造邻接。
        features, adjacency, permutation = permute_graph(shared_features, adjacency, 1200 + pair_index)  # 对同一配对使用完全相同的节点置换。
        graph_id = f"UN-{pair_index + 1}{'N' if label == 0 else 'R'}"  # 构造未见结构图编号。
        unseen_graphs.append({"id": graph_id, "pair": pair_index, "features": features, "adjacency": adjacency, "label": label, "topology": specification["name"]})  # 保存未见结构与人工标签。
def stack_graphs(graph_list):  # 把图记录列表堆叠成批量张量。
    feature_batch = torch.stack([graph["features"] for graph in graph_list])  # 堆叠节点特征。
    adjacency_batch = torch.stack([graph["adjacency"] for graph in graph_list])  # 堆叠邻接矩阵。
    target_batch = torch.tensor([graph["label"] for graph in graph_list])  # 堆叠图级人工标签。
    return feature_batch, adjacency_batch, target_batch  # 返回模型训练或评估需要的三个张量。
train_features, train_adjacency, train_targets = stack_graphs(train_graphs)  # 构造十二图训练批次。
template_features, template_adjacency, template_targets = stack_graphs(template_graphs)  # 构造六图模板重排批次。
unseen_features, unseen_adjacency, unseen_targets = stack_graphs(unseen_graphs)  # 构造八图未见结构批次。
unseen_pair_feature_error = max(float((unseen_graphs[index]["features"] - unseen_graphs[index + 1]["features"]).abs().max()) for index in range(0, len(unseen_graphs), 2))  # 验证每对标签图的节点特征完全相同。
unseen_pair_edge_count_match = all(int(unseen_graphs[index]["adjacency"].sum()) == int(unseen_graphs[index + 1]["adjacency"].sum()) for index in range(0, len(unseen_graphs), 2))  # 验证每对标签图的总边数完全相同。
print("集合       图号   标签         边数  风险均值  高风险直连  拓扑说明")  # 输出三套数据的结构预览表头。
preview_graphs = train_graphs[:4] + template_graphs + unseen_graphs  # 选择部分训练图和全部两类留出图展示。
for graph in preview_graphs:  # 遍历需要展示的图记录。
    high_mask = graph["features"][:, 0] > 0.7  # 根据独立风险字段找到高风险账户。
    high_edges = int(graph["adjacency"][high_mask][:, high_mask].sum().item() / 2)  # 统计高风险账户之间的无向边。
    edge_count = int(graph["adjacency"].sum().item() / 2)  # 统计当前图的无向边总数。
    split_name = "训练" if graph["id"].startswith("TR") else ("模板重排" if graph["id"].startswith("TP") else "未见结构")  # 根据编号恢复数据集合名称。
    print(f"{split_name:<10} {graph['id']:<6} {label_names[graph['label']]:<10} {edge_count:>3}    {float(graph['features'][:, 0].mean()):.3f}       {high_edges:>2}      {graph['topology']}")  # 展示结构差异而不暴露标签字段。
print(f"未见配对特征最大差={unseen_pair_feature_error:.1f}，每对边数一致={unseen_pair_edge_count_match}")  # 直接展示特征与总边数没有标签泄漏。

集合       图号   标签         边数  风险均值  高风险直连  拓扑说明
训练         TR-1N  无高风险闭环       6    0.497        0      训练模板重排
训练         TR-1R  高风险闭环        6    0.497        3      训练模板重排
训练         TR-2N  无高风险闭环       6    0.497        0      训练模板重排
训练         TR-2R  高风险闭环        6    0.497        3      训练模板重排
模板重排       TP-1N  无高风险闭环       6    0.497        0      训练模板重排
模板重排       TP-1R  高风险闭环        6    0.497        3      训练模板重排
模板重排       TP-2N  无高风险闭环       6    0.497        0      训练模板重排
模板重排       TP-2R  高风险闭环        6    0.497        3      训练模板重排
模板重排       TP-3N  无高风险闭环       6    0.497        0      训练模板重排
模板重排       TP-3R  高风险闭环        6    0.497        3      训练模板重排
未见结构       UN-1N  无高风险闭环       5    0.497        0      五边低风险桥/三角尾链
未见结构       UN-1R  高风险闭环        5    0.497        3      五边低风险桥/三角尾链
未见结构       UN-2N  无高风险闭环       7    0.497        2      七边高风险路径/三角多桥
未见结构       UN-2R  高风险闭环        7    0.497        3      七边高风险路径/三角多桥
未见结构       UN-3N  无高风险闭环       6    0.590        

## Baseline（基线）：固定风险均值阈值

最便宜的图级基线对节点风险直接 mean pooling，并使用业务预先约定的 0.5 阈值。阈值在训练和测试结果出现前就固定，不会查看模板重排或未见结构标签。每个正常/闭环配对拥有相同节点特征，所以这个基线在两套测试上都无法区分关系结构。

In [2]:
risk_threshold = 0.5  # 在任何模型训练和测试评估前固定业务风险均值阈值。
def evaluate_mean_baseline(features, targets):  # 在指定图批次上评估无结构风险均值方案。
    scores = features[:, :, 0].mean(dim=1)  # 对每张图的节点风险直接取平均。
    predictions = (scores >= risk_threshold).long()  # 使用预先固定的零点五阈值生成类别。
    accuracy = float((predictions == targets).float().mean())  # 计算当前集合的图级准确率。
    return scores, predictions, accuracy  # 返回逐图分数、预测和汇总指标。
template_baseline_scores, template_baseline_predictions, template_baseline_accuracy = evaluate_mean_baseline(template_features, template_targets)  # 评估同模板重排集合。
unseen_baseline_scores, unseen_baseline_predictions, unseen_baseline_accuracy = evaluate_mean_baseline(unseen_features, unseen_targets)  # 评估真正未见结构集合。
print("未见图号  gold         平均风险  基线预测      正确")  # 输出未见结构逐图基线表头。
for row, graph in enumerate(unseen_graphs):  # 遍历八张真正未见结构图。
    predicted_name = label_names[int(unseen_baseline_predictions[row])]  # 还原均值阈值预测名称。
    print(f"{graph['id']:<8} {label_names[graph['label']]:<10} {float(unseen_baseline_scores[row]):.3f}     {predicted_name:<10} {bool(unseen_baseline_predictions[row] == unseen_targets[row])}")  # 展示同特征配对得到同一基线分数。
print(f"风险均值 Baseline：模板重排={template_baseline_accuracy:.1%}，未见结构={unseen_baseline_accuracy:.1%}")  # 分开报告两套测试指标。
print("阈值来源：训练前固定为 0.5，没有读取任何测试标签。")  # 明确排除测试集调阈值。

未见图号  gold         平均风险  基线预测      正确
UN-1N    无高风险闭环     0.497     无高风险闭环     True
UN-1R    高风险闭环      0.497     无高风险闭环     False
UN-2N    无高风险闭环     0.497     无高风险闭环     True
UN-2R    高风险闭环      0.497     无高风险闭环     False
UN-3N    无高风险闭环     0.590     高风险闭环      False
UN-3R    高风险闭环      0.590     高风险闭环      True
UN-4N    无高风险闭环     0.497     无高风险闭环     True
UN-4R    高风险闭环      0.497     无高风险闭环     False
风险均值 Baseline：模板重排=50.0%，未见结构=50.0%
阈值来源：训练前固定为 0.5，没有读取任何测试标签。


## 核心实现：消息传递、软分配与图粗化

先给邻接矩阵加自环并做 $D^{-1/2}(A+I)D^{-1/2}$。embedding 分支生成节点内容，pool 分支生成每个节点到两个簇的概率；`transpose(1, 2) @` 明确实现 $S^TZ$ 与 $S^TAS$。分类头读取两个簇的表示以及 2×2 粗化邻接，避免再次把层次结构完全平均掉。

In [3]:
def normalize_adjacency(adjacency):  # 对批量邻接矩阵加自环并做对称归一化。
    node_count = adjacency.shape[-1]  # 读取每张图的节点数。
    identity = torch.eye(node_count, device=adjacency.device).unsqueeze(0)  # 创建可广播的单位矩阵。
    with_self_loops = adjacency + identity  # 加入节点自己的消息通道。
    degrees = with_self_loops.sum(dim=-1).clamp_min(1.0)  # 计算度并防止孤立节点除零。
    inverse_sqrt = degrees.pow(-0.5)  # 计算每个节点的度负二分之一次方。
    return inverse_sqrt.unsqueeze(-1) * with_self_loops * inverse_sqrt.unsqueeze(-2)  # 返回对称归一化邻接。
class ManualDiffPool(nn.Module):  # 定义不用图框架的两簇 DiffPool 分类器。
    def __init__(self, input_dim=2, hidden_dim=10, cluster_count=2, class_count=2):  # 初始化消息、分配、粗化与分类参数。
        super().__init__()  # 注册基础模块状态。
        self.cluster_count = cluster_count  # 保存目标簇数量。
        self.input_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * math.sqrt(2.0 / input_dim))  # 创建第一层节点消息权重。
        self.assignment_weight = nn.Parameter(torch.randn(hidden_dim, cluster_count) * 0.18)  # 创建节点到簇的分配权重。
        self.embedding_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.18)  # 创建节点内容表示权重。
        self.cluster_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.18)  # 创建粗化图消息传递权重。
        readout_dim = cluster_count * hidden_dim + cluster_count * cluster_count  # 计算保留簇顺序和粗化边的读出维度。
        self.output_weight = nn.Parameter(torch.randn(readout_dim, class_count) * 0.12)  # 创建图表示到类别的权重。
        self.output_bias = nn.Parameter(torch.zeros(class_count))  # 创建类别偏置。
    def forward(self, features, adjacency, return_details=False):  # 执行完整 DiffPool 前向计算。
        normalized = normalize_adjacency(adjacency)  # 获取节点级归一化邻接。
        hidden = torch.relu(normalized @ features @ self.input_weight)  # 汇聚邻居原始特征并映射到隐藏维。
        assignment_logits = normalized @ hidden @ self.assignment_weight  # 用 pool 分支计算节点到簇分数。
        assignments = torch.softmax(assignment_logits, dim=-1)  # 让每个节点在两个簇上的概率和为一。
        node_embeddings = torch.relu(normalized @ hidden @ self.embedding_weight)  # 用 embed 分支计算待汇聚节点内容。
        pooled_features = assignments.transpose(1, 2) @ node_embeddings  # 显式实现簇特征 X'=S^TZ。
        pooled_adjacency = assignments.transpose(1, 2) @ adjacency @ assignments  # 显式实现粗化邻接 A'=S^TAS。
        normalized_pooled = normalize_adjacency(pooled_adjacency)  # 对簇级邻接再次加自环归一化。
        cluster_embeddings = torch.relu(normalized_pooled @ pooled_features @ self.cluster_weight)  # 在粗化图上再传播一次消息。
        readout = torch.cat([cluster_embeddings.reshape(features.shape[0], -1), pooled_adjacency.reshape(features.shape[0], -1)], dim=-1)  # 拼接全部簇内容与簇间结构。
        logits = readout @ self.output_weight + self.output_bias  # 计算正常与团伙两类分数。
        if return_details:  # 教学观察模式需要返回软分配和粗化结果。
            return logits, assignments, pooled_features, pooled_adjacency, assignment_logits  # 暴露 DiffPool 的关键中间量。
        return logits  # 普通训练模式只返回分类分数。
torch.manual_seed(67)  # 固定 DiffPool 参数初始化。
model = ManualDiffPool()  # 创建两簇层次化图分类器。
preview_logits, preview_assignments, preview_features, preview_adjacency, preview_assignment_logits = model(train_features[:2], train_adjacency[:2], return_details=True)  # 对两张训练图执行前向。
print("节点特征 / 邻接 / S / X' / A'：", tuple(train_features[:2].shape), tuple(train_adjacency[:2].shape), tuple(preview_assignments.shape), tuple(preview_features.shape), tuple(preview_adjacency.shape))  # 展示每个公式对应的真实形状。
print("首图每节点到两簇的概率：")  # 输出软聚类矩阵标题。
for node_index, probabilities in enumerate(preview_assignments[0]):  # 遍历首图六个账户。
    print(f"节点{node_index}: {[round(float(value), 3) for value in probabilities]}")  # 展示节点分配而非只检查 shape。
print("首图粗化邻接 A'：", [[round(float(value), 3) for value in row] for row in preview_adjacency[0]])  # 展示簇内与簇间交易强度。

节点特征 / 邻接 / S / X' / A'： (2, 6, 2) (2, 6, 6) (2, 6, 2) (2, 2, 10) (2, 2, 2)
首图每节点到两簇的概率：
节点0: [0.482, 0.518]
节点1: [0.482, 0.518]
节点2: [0.464, 0.536]
节点3: [0.471, 0.529]
节点4: [0.47, 0.53]
节点5: [0.47, 0.53]
首图粗化邻接 A'： [[2.656, 2.99], [2.99, 3.365]]


## 真实训练、置换探针与两类留出结果

分类 loss 之外加入很小的簇负载均衡项，防止所有节点长期挤进一个簇。模型只在 12 张训练模板图上更新参数。训练结束后先把一张图再次随机置换并比较 logits，专门验证置换不变性；再分别报告“训练模板的新重排”和“真正未见结构”准确率及逐图概率。分类阈值固定为 0.5，不根据任何测试标签调整。最后展示一张未见闭环图的 S、X'=S^TZ 和 A'=S^TAS 中间结果。

In [4]:
decision_threshold = 0.5  # 在评估前固定二分类概率阈值而不查看任何测试标签。
optimizer = torch.optim.Adam(model.parameters(), lr=0.025)  # 创建更新全部手写参数的 Adam 优化器。
history = []  # 保存分类 loss、均衡 loss、训练准确率与梯度范数。
for epoch in range(260):  # 只在十二张训练模板图上执行多轮真实优化。
    optimizer.zero_grad()  # 清除上一轮累计梯度。
    logits, assignments, _, _, _ = model(train_features, train_adjacency, return_details=True)  # 前向得到分类和节点软分配。
    classification_loss = F.cross_entropy(logits, train_targets)  # 只用训练图标签计算图级二分类交叉熵。
    cluster_load = assignments.mean(dim=1)  # 计算每张训练图两个簇的平均负载。
    balance_loss = ((cluster_load - 0.5) ** 2).mean()  # 惩罚长期空簇与单簇塌缩。
    loss = classification_loss + 0.08 * balance_loss  # 合并主任务与轻量结构正则。
    loss.backward()  # 把图分类误差传播到 assignment 与 embedding 分支。
    gradient_norm = float(model.assignment_weight.grad.norm().detach())  # 观察分配网络确实获得梯度。
    optimizer.step()  # 按真实梯度更新模型。
    train_accuracy = float((logits.argmax(dim=-1) == train_targets).float().mean().detach())  # 计算当前训练图准确率。
    history.append((float(classification_loss.detach()), float(balance_loss.detach()), train_accuracy, gradient_norm))  # 保存完整优化轨迹。
probe_permutation = torch.tensor([2, 5, 0, 4, 1, 3])  # 定义与训练排列不同的确定性节点重排。
probe_features = train_features[:1]  # 取一张训练图作为置换不变性探针。
probe_adjacency = train_adjacency[:1]  # 取同一张图的邻接矩阵。
permuted_probe_features = probe_features[:, probe_permutation]  # 按新编号重排节点特征。
permuted_probe_adjacency = probe_adjacency[:, probe_permutation][:, :, probe_permutation]  # 同步重排邻接行列。
with torch.no_grad():  # 关闭三套评估的梯度记录。
    original_probe_logits = model(probe_features, probe_adjacency)  # 计算原节点编号下的图级 logits。
    permuted_probe_logits = model(permuted_probe_features, permuted_probe_adjacency)  # 计算再次重排后的图级 logits。
    template_logits, template_assignments, template_pooled_features, template_pooled_adjacency, template_assignment_logits = model(template_features, template_adjacency, return_details=True)  # 获取模板重排集合的完整输出。
    unseen_logits, unseen_assignments, unseen_pooled_features, unseen_pooled_adjacency, unseen_assignment_logits = model(unseen_features, unseen_adjacency, return_details=True)  # 获取真正未见结构集合的完整输出。
permutation_logit_error = float((original_probe_logits - permuted_probe_logits).abs().max())  # 计算只改变节点编号后的最大 logits 偏差。
template_probabilities = torch.softmax(template_logits, dim=-1)  # 把模板重排 logits 转为类别概率。
unseen_probabilities = torch.softmax(unseen_logits, dim=-1)  # 把未见结构 logits 转为类别概率。
template_predictions = (template_probabilities[:, 1] >= decision_threshold).long()  # 使用固定阈值生成模板重排预测。
unseen_predictions = (unseen_probabilities[:, 1] >= decision_threshold).long()  # 使用同一固定阈值生成未见结构预测。
template_accuracy = float((template_predictions == template_targets).float().mean())  # 计算同模板新重排准确率。
unseen_accuracy = float((unseen_predictions == unseen_targets).float().mean())  # 计算真正未见拓扑准确率。
print("阶段       分类loss  均衡loss  训练准确率  assignment梯度")  # 输出真实训练轨迹表头。
print(f"首轮       {history[0][0]:.4f}     {history[0][1]:.4f}      {history[0][2]:.1%}       {history[0][3]:.4f}")  # 展示随机初始化阶段。
print(f"末轮       {history[-1][0]:.4f}     {history[-1][1]:.4f}      {history[-1][2]:.1%}       {history[-1][3]:.4f}")  # 展示训练模板上的优化结果。
print(f"显式节点置换 logits 最大差：{permutation_logit_error:.8f}")  # 单独报告不变量而不把它称为结构泛化。
print("模板重排图 | gold         DiffPool预测  闭环概率  正确")  # 输出训练拓扑新重排结果表头。
for row, graph in enumerate(template_graphs):  # 遍历六张模板重排图。
    predicted_name = label_names[int(template_predictions[row])]  # 还原当前模板图的预测名称。
    print(f"{graph['id']:<10} | {label_names[graph['label']]:<10} {predicted_name:<10} {float(template_probabilities[row, 1]):.4f}   {bool(template_predictions[row] == template_targets[row])}")  # 展示同模板稳定性。
print("未见结构图 | 边数 | gold         DiffPool预测  闭环概率  正确 | 结构")  # 输出真正未见拓扑结果表头。
for row, graph in enumerate(unseen_graphs):  # 遍历八张未见结构图。
    predicted_name = label_names[int(unseen_predictions[row])]  # 还原当前未见图的预测名称。
    edge_count = int(graph["adjacency"].sum().item() / 2)  # 读取当前未见图的真实边数。
    print(f"{graph['id']:<10} | {edge_count:>2} | {label_names[graph['label']]:<10} {predicted_name:<10} {float(unseen_probabilities[row, 1]):.4f}   {bool(unseen_predictions[row] == unseen_targets[row])} | {graph['topology']}")  # 展示模型在新边数和新闭环上的真实表现。
print(f"分项准确率：模板重排 baseline={template_baseline_accuracy:.1%} / DiffPool={template_accuracy:.1%}；未见结构 baseline={unseen_baseline_accuracy:.1%} / DiffPool={unseen_accuracy:.1%}")  # 严格分开报告两套指标。
focus_row = 1  # 选择第一张真正未见的五边闭环图观察层次结构。
print("未见闭环图簇负载：", [round(float(value), 3) for value in unseen_assignments[focus_row].sum(dim=0)])  # 展示六节点如何分到两个簇。
print("未见闭环图 X'=S^TZ 形状：", tuple(unseen_pooled_features[focus_row].shape))  # 展示软聚类后的真实簇特征形状。
print("未见闭环图 A'=S^TAS：", [[round(float(value), 3) for value in row] for row in unseen_pooled_adjacency[focus_row]])  # 展示未见图的簇级连接。

阶段       分类loss  均衡loss  训练准确率  assignment梯度
首轮       0.7019     0.0006      25.0%       0.0388
末轮       0.0000     0.0025      100.0%       0.0002
显式节点置换 logits 最大差：0.00000191
模板重排图 | gold         DiffPool预测  闭环概率  正确
TP-1N      | 无高风险闭环     无高风险闭环     0.0000   True
TP-1R      | 高风险闭环      高风险闭环      1.0000   True
TP-2N      | 无高风险闭环     无高风险闭环     0.0000   True
TP-2R      | 高风险闭环      高风险闭环      1.0000   True
TP-3N      | 无高风险闭环     无高风险闭环     0.0000   True
TP-3R      | 高风险闭环      高风险闭环      1.0000   True
未见结构图 | 边数 | gold         DiffPool预测  闭环概率  正确 | 结构
UN-1N      |  5 | 无高风险闭环     无高风险闭环     0.1871   True | 五边低风险桥/三角尾链
UN-1R      |  5 | 高风险闭环      高风险闭环      1.0000   True | 五边低风险桥/三角尾链
UN-2N      |  7 | 无高风险闭环     无高风险闭环     0.1355   True | 七边高风险路径/三角多桥
UN-2R      |  7 | 高风险闭环      高风险闭环      0.5632   True | 七边高风险路径/三角多桥
UN-3N      |  6 | 无高风险闭环     无高风险闭环     0.1155   True | 六边四高路径/四节点闭环
UN-3R      |  6 | 高风险闭环      高风险闭环      0.8915   True | 六边四高路径/四节点闭环
UN-4N      |  8 | 无高风险闭

## 结果解读

模板重排准确率是 100%，回答的是“节点编号和轻微金额扰动改变后，模型是否仍识别训练见过的两种拓扑”；显式置换 logits 最大差只有 0.00000191，进一步隔离了编号不变量。真正未见结构准确率是 87.5%，明显低于模板重排：模型正确识别了五边三角尾链、七边三角多桥和训练未见的四节点闭环，却把八边稠密三角 UN-4R 判成正常，闭环概率接近 0。这说明训练中的单一六边三角模板不足以覆盖“高风险闭环与稠密低风险子图同时存在”的形态，不能用模板重排高分冒充结构泛化。输出中的 0.5 分类阈值在测试前固定，没有用任一测试标签调参，也没有为了修正 UN-4R 偷调阈值。每对未见图的特征与边数都相同，模型只能依靠消息传递、S^TZ 与 S^TAS 中的关系组合做决策。两个簇的编号可交换，不能把“簇 0”硬解释成固定业务角色。

## 失败案例：assignment 被路由偏置压成单簇

若某个簇初始 bias 极大，普通 row-softmax 会让所有节点进入同一簇，另一个簇几乎没有梯度和信息。下面在真正未见闭环图的 assignment logits 上人为加入偏置复现塌缩，再用交替行/列归一化的简化 Sinkhorn 把总负载拉回均衡；这只是修复手段演示，生产中还应同时监控熵与簇占用。

In [5]:
collapsed_logits = unseen_assignment_logits[focus_row].detach() + torch.tensor([5.0, -5.0])  # 给第一簇加入巨大偏置以复现路由塌缩。
collapsed_assignment = torch.softmax(collapsed_logits, dim=-1)  # 直接 row-softmax 得到几乎单簇分配。
transport = torch.exp(collapsed_logits - collapsed_logits.max())  # 把偏置 logits 转成稳定的正权重矩阵。
for _ in range(20):  # 交替规范化节点行和簇列以逼近均衡分配。
    transport = transport / transport.sum(dim=1, keepdim=True).clamp_min(1e-9)  # 让每个节点的分配和为一。
    transport = transport / transport.sum(dim=0, keepdim=True).clamp_min(1e-9) * (transport.shape[0] / transport.shape[1])  # 让每个簇总负载接近三节点。
balanced_assignment = transport / transport.sum(dim=1, keepdim=True).clamp_min(1e-9)  # 最后恢复每个节点概率和为一。
collapsed_load = collapsed_assignment.sum(dim=0)  # 统计错误 softmax 的两个簇负载。
balanced_load = balanced_assignment.sum(dim=0)  # 统计 Sinkhorn 修复后的两个簇负载。
collapsed_pooled_adjacency = collapsed_assignment.transpose(0, 1) @ unseen_adjacency[focus_row] @ collapsed_assignment  # 计算塌缩后的簇级邻接。
balanced_pooled_adjacency = balanced_assignment.transpose(0, 1) @ unseen_adjacency[focus_row] @ balanced_assignment  # 计算均衡后的簇级邻接。
print("方案             簇0负载  簇1负载  A'[1,1]")  # 输出塌缩与修复对比表头。
print(f"偏置后softmax      {float(collapsed_load[0]):.3f}    {float(collapsed_load[1]):.3f}     {float(collapsed_pooled_adjacency[1, 1]):.6f}")  # 展示空簇几乎没有结构。
print(f"Sinkhorn均衡       {float(balanced_load[0]):.3f}    {float(balanced_load[1]):.3f}     {float(balanced_pooled_adjacency[1, 1]):.6f}")  # 展示修复后两簇都承载节点与边。
print("修复后的前两节点分配：", [[round(float(value), 3) for value in row] for row in balanced_assignment[:2]])  # 展示修复并非硬编码节点标签。

方案             簇0负载  簇1负载  A'[1,1]
偏置后softmax      6.000    0.000     0.000000
Sinkhorn均衡       3.000    3.000     2.145513
修复后的前两节点分配： [[0.732, 0.268], [0.179, 0.821]]


## 生产差距与追问

真实图会有变长节点数、方向/类型/时间边、极端类别不均衡和跨时间泄漏；必须用 padding mask 或稀疏算子，按事件时间构建子图，并按未来时间、全新账户群和新型拓扑共同留出。模板重排、随机边扰动和真正未见团伙形态应分别报告，不能把同模板增强当作独立测试。生产评估还要报告 PR-AUC、团伙召回、误杀成本、拓扑切片和置信度校准，任何阈值只能在训练期校准集冻结。DiffPool 的 S^TAS 在大图上是内存瓶颈，可改用分层采样、稀疏聚类或离线社区先验。辅助 link prediction loss 也要避免迫使模型重建噪声边。

## 最小回归测试

In [6]:
assert len(train_graphs) >= 5 and len(unseen_graphs) == 8  # 保证训练和真正未见结构集合都有足够可读样本。
assert template_accuracy > template_baseline_accuracy  # 保护模型在训练拓扑新重排上确实优于无结构均值基线。
assert decision_threshold == 0.5  # 保护测试决策使用预先固定阈值而非读取标签调参。
assert unseen_pair_feature_error == 0.0 and unseen_pair_edge_count_match  # 保护未见配对没有节点特征或总边数标签泄漏。
assert permutation_logit_error < 1e-5  # 保护显式节点置换不会改变图级语义输出。
assert tuple(unseen_pooled_adjacency.shape[1:]) == (2, 2)  # 保护 S 转置粗化公式在未见结构上仍得到两簇邻接。
assert float(balanced_load.min()) > 2.5 and float(collapsed_load.min()) < 0.01  # 保护单簇失败与 Sinkhorn 修复都真实发生。
print("最小回归测试通过：无泄漏配对、置换不变、分项评估、图粗化与 assignment 修复均成立。")  # 输出集中测试结论。

最小回归测试通过：无泄漏配对、置换不变、分项评估、图粗化与 assignment 修复均成立。
